# scSemiProfiler — the same rare-population and disease-DEG metrics as PBISC floor (2026-10-30)
Inputs: scSemi strict_5pairs draw0 export from three_model_20261022. Pseudobulk = rate sums per sample × own six-type label. Cells = one Poisson draw.
kNN transfer labels from real half A come from the same pairwise Harmony pipeline as floor (big_joint.py). Note that scSemi iLISI is 1.00, so the integration failed and the transfer is unreliable.
Transfer-free rare estimates come from three_model claims_rare.py.
Designs:
- **full**: generator vs all real samples.
- **paired**: real restricted to the *same* samples that scSemi regenerates.
- **nonA**: scSemi samples whose patient is not in real A.

In [ ]:
import os, sys; os.chdir('/home/claude/pbisc'); sys.path.insert(0,'/home/claude/pbisc')
import pandas as pd; pd.set_option('display.width',250); pd.set_option('display.max_columns',30)

## 1. Disease DEG, modules, between-sample SD

In [ ]:
exec(open('scsemi/s1_deg.py').read())

## 2. Rare populations: yield and marker score (kNN transfer)

In [ ]:
exec(open('scsemi/s2_rare.py').read())

### Transfer-free rare estimates (three_model claims_rare)

In [ ]:
for coh in ['sepsis','covid']:
    t=pd.read_csv(f'/mnt/user-data/uploads/integration_report_20261012/three_model_20261022/results/{coh}_rare_transfer_free.csv')
    t['ratio']=t.gen_estimate_pct/t.realA_cv_estimate_pct
    p=t.pivot_table(index='subset',columns='model_depth',values='ratio'); p['realB']=(t.realB_estimate_pct/t.realA_cv_estimate_pct).groupby(t.subset).first(); print(coh); display(p.round(2))

## 3. Top10-marker de novo annotation (run as script `scsemi/s3_markerann.py`)
```python
"""Top10-marker de novo annotation of scSemiProfiler cells (same procedure as markerann/s2_annotate.py), then share error and identity r vs real A."""
import sys; sys.path.insert(0,"/home/claude/pbisc/markerann")
import numpy as np, pandas as pd, anndata as ad, scanpy as sc, warnings; warnings.filterwarnings("ignore")
from annot_lib import annotate
SS="/mnt/user-data/uploads/integration_report_20261012/three_model_20261022/data/"
for coh in ["sepsis","covid"]:
    a=ad.read_h5ad(f"{SS}{coh}_scsemi.h5ad"); a.obs_names_make_unique()
    tr=pd.read_csv(f"/mnt/user-data/uploads/integration_report_20261012/ilisi_floor_20261021/results/big/{coh}_scsemi_transfer.csv.gz",index_col=0,keep_default_na=False)
    a.obs["six_knn"]=tr.six_tr.values; a.obs["fine_knn"]=tr.fine_tr.values
    sc.pp.normalize_total(a,1e4); sc.pp.log1p(a)
    a=annotate(a,coh)
    cols=["sample","patient","group","six_type","aifi_l2","six_knn","fine_knn","cl1","cl2","six_mk_cl","fine_mk_cl","six_mk_cell","fine_mk_cell"]
    a.obs[cols].to_csv(f"scsemi/out/{coh}_scsemi_markerann_labels.csv.gz")
    a.write_h5ad(f"scsemi/out/{coh}_scsemi_annot.h5ad",compression="gzip")
    print(coh,"L1",a.obs.cl1.nunique(),"L2",a.obs.cl2.nunique(),flush=True)

```

In [ ]:
exec(open('scsemi/s5_markerann_compare.py').read())

## 4. Specificity law

In [ ]:
exec(open('scsemi/s4_law.py').read())
import numpy as np
G=pd.read_csv('scsemi/out/law_gene_level.csv.gz'); print('gene x type with zero scSemi expression:',round((G.scsemi_ratio==0).mean(),3))

## 5. Summary figure

In [ ]:
exec(open('scsemi/fig_scsemi.py').read())
from IPython.display import Image; display(Image('scsemi/out/fig_scsemi_vs_floor.png',width=1100))